In [14]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import pickle
import warnings
warnings.filterwarnings('ignore')

In [15]:
# Load data
fixtures = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fixtures_clean.csv")
groups = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
elo = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams.csv")
ranks = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fifa_rankings.csv")

# Load models
with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model.pkl", 'rb') as f:
    home_model = pickle.load(f)

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model.pkl", 'rb') as f:
    away_model = pickle.load(f)

print("All files loaded!")
print(f"Fixtures: {len(fixtures)} matches")
print(f"Groups: {len(groups)} teams")

All files loaded!
Fixtures: 72 matches
Groups: 48 teams


In [16]:
# Merge elo and rankings into one lookup
team_info = elo.merge(ranks, left_on='team', right_on='Nation', how='left').drop(columns='Nation')
team_info = team_info.rename(columns={'FIFA_2026_rank': 'fifa_rank'})
team_info = team_info[['team', 'elo', 'fifa_rank']]

# Convert to dictionary for fast lookup
team_dict = team_info.set_index('team').to_dict(orient='index')

# Test it
print(team_dict['Spain'])
print(team_dict['Argentina'])

{'elo': 1961.161784240624, 'fifa_rank': 2.0}
{'elo': 1857.067387602092, 'fifa_rank': 3.0}


In [17]:
def predict_match(home_team, away_team, neutral=True, tournament_weight=5.0):
    home_elo = team_dict[home_team]['elo']
    away_elo = team_dict[away_team]['elo']
    home_rank = team_dict[home_team]['fifa_rank']
    away_rank = team_dict[away_team]['fifa_rank']

    match = pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'neutral': int(neutral),
        'tournament_weight': tournament_weight,
        'fifa_rank_diff': home_rank - away_rank
    }])

    home_xg = home_model.predict(match)[0]
    away_xg = away_model.predict(match)[0]

    return home_xg, away_xg


def match_probabilities(home_xg, away_xg, max_goals=10):
    home_win, draw, away_win = 0, 0, 0
    for i in range(max_goals):
        for j in range(max_goals):
            p = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)
            if i > j:
                home_win += p
            elif i == j:
                draw += p
            else:
                away_win += p
    return home_win, draw, away_win

# Test
home_xg, away_xg = predict_match('Spain', 'Argentina')
hw, d, aw = match_probabilities(home_xg, away_xg)
print(f"Spain vs Argentina")
print(f"Spain win: {hw*100:.1f}%  Draw: {d*100:.1f}%  Argentina win: {aw*100:.1f}%")

Spain vs Argentina
Spain win: 49.9%  Draw: 26.6%  Argentina win: 23.4%


In [18]:
def simulate_match(home_team, away_team, neutral=True, tournament_weight=5.0):
    home_xg, away_xg = predict_match(home_team, away_team, neutral, tournament_weight)
    
    # Randomly sample goals from Poisson distribution
    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)
    
    return home_goals, away_goals

# Test it a few times
for i in range(5):
    hg, ag = simulate_match('Spain', 'Argentina')
    print(f"Spain {hg} - {ag} Argentina")

Spain 2 - 0 Argentina
Spain 1 - 2 Argentina
Spain 0 - 2 Argentina
Spain 2 - 2 Argentina
Spain 0 - 0 Argentina


In [19]:
def simulate_group(group_name, group_teams, fixtures_df):
    # Set up points table
    table = {team: {'W': 0, 'D': 0, 'L': 0, 'GF': 0, 'GA': 0, 'Pts': 0} for team in group_teams}
    
    # Get fixtures for this group
    group_fixtures = fixtures_df[
        fixtures_df['home_team'].isin(group_teams) & 
        fixtures_df['away_team'].isin(group_teams)
    ]
    
    # Simulate each match
    for _, row in group_fixtures.iterrows():
        home = row['home_team']
        away = row['away_team']
        neutral = row['neutral']
        
        hg, ag = simulate_match(home, away, neutral=neutral)
        
        # Update goals
        table[home]['GF'] += hg
        table[home]['GA'] += ag
        table[away]['GF'] += ag
        table[away]['GA'] += hg
        
        # Update points
        if hg > ag:
            table[home]['W'] += 1
            table[home]['Pts'] += 3
            table[away]['L'] += 1
        elif hg < ag:
            table[away]['W'] += 1
            table[away]['Pts'] += 3
            table[home]['L'] += 1
        else:
            table[home]['D'] += 1
            table[away]['D'] += 1
            table[home]['Pts'] += 1
            table[away]['Pts'] += 1
    
    # Convert to dataframe and sort
    table_df = pd.DataFrame(table).T
    table_df['GD'] = table_df['GF'] - table_df['GA']
    table_df = table_df.sort_values(['Pts', 'GD', 'GF'], ascending=False)
    table_df.index.name = 'team'
    table_df = table_df.reset_index()
    
    return table_df

# Test with Group A
group_a_teams = groups[groups['group'] == 'A']['nation'].tolist()
print(f"Group A teams: {group_a_teams}")
result = simulate_group('A', group_a_teams, fixtures)
print(result[['team', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']])

Group A teams: ['Mexico', 'South Africa', 'South Korea', 'Czech Republic']
             team  W  D  L  GF  GA  GD  Pts
0          Mexico  2  1  0   7   0   7    7
1     South Korea  2  1  0   5   0   5    7
2    South Africa  1  0  2   3   6  -3    3
3  Czech Republic  0  0  3   2  11  -9    0


In [20]:
def simulate_all_groups(groups_df, fixtures_df):
    all_tables = {}
    
    for group_name in sorted(groups_df['group'].unique()):
        group_teams = groups_df[groups_df['group'] == group_name]['nation'].tolist()
        table = simulate_group(group_name, group_teams, fixtures_df)
        all_tables[group_name] = table
    
    return all_tables

# Test
all_tables = simulate_all_groups(groups, fixtures)

# Print all groups
for group_name, table in all_tables.items():
    print(f"\nGroup {group_name}")
    print(table[['team', 'W', 'D', 'L', 'GD', 'Pts']].to_string(index=False))


Group A
          team  W  D  L  GD  Pts
        Mexico  2  1  0   4    7
Czech Republic  2  0  1   1    6
   South Korea  0  2  1  -1    2
  South Africa  0  1  2  -4    1

Group B
                  team  W  D  L  GD  Pts
                Canada  2  1  0   7    7
Bosnia and Herzegovina  1  1  1  -3    4
                 Qatar  1  0  2  -3    3
           Switzerland  0  2  1  -1    2

Group C
    team  W  D  L  GD  Pts
  Brazil  2  1  0   2    7
 Morocco  1  2  0   2    5
Scotland  1  1  1   0    4
   Haiti  0  0  3  -4    0

Group D
         team  W  D  L  GD  Pts
United States  2  1  0   3    7
    Australia  1  1  1   1    4
     Paraguay  1  1  1   0    4
       Turkey  0  1  2  -4    1

Group E
       team  W  D  L  GD  Pts
    Ecuador  2  1  0   4    7
    Germany  1  2  0   1    5
    Curaçao  1  1  1  -2    4
Ivory Coast  0  0  3  -3    0

Group F
       team  W  D  L  GD  Pts
    Tunisia  3  0  0   3    9
      Japan  2  0  1   4    6
Netherlands  1  0  2   1    3
     Sweden

In [21]:
def get_qualifiers(all_tables):
    group_winners = []
    group_runners_up = []
    third_placed = []

    for group_name, table in all_tables.items():
        group_winners.append({
            'team': table.iloc[0]['team'],
            'group': group_name,
            'pts': table.iloc[0]['Pts'],
            'gd': table.iloc[0]['GD'],
            'gf': table.iloc[0]['GF']
        })
        group_runners_up.append({
            'team': table.iloc[1]['team'],
            'group': group_name,
            'pts': table.iloc[1]['Pts'],
            'gd': table.iloc[1]['GD'],
            'gf': table.iloc[1]['GF']
        })
        third_placed.append({
            'team': table.iloc[2]['team'],
            'group': group_name,
            'pts': table.iloc[2]['Pts'],
            'gd': table.iloc[2]['GD'],
            'gf': table.iloc[2]['GF']
        })

    third_df = pd.DataFrame(third_placed)
    third_df = third_df.sort_values(['pts', 'gd', 'gf'], ascending=False).head(8)

    qualifiers = (
        [t['team'] for t in group_winners] +
        [t['team'] for t in group_runners_up] +
        third_df['team'].tolist()
    )

    return qualifiers, group_winners, group_runners_up, third_df

In [22]:
def get_qualifiers(all_tables):
    group_winners = []
    group_runners_up = []
    third_placed = []

    for group_name, table in all_tables.items():
        group_winners.append({
            'team': table.iloc[0]['team'],
            'group': group_name,
            'pts': table.iloc[0]['Pts'],
            'gd': table.iloc[0]['GD'],
            'gf': table.iloc[0]['GF']
        })
        group_runners_up.append({
            'team': table.iloc[1]['team'],
            'group': group_name,
            'pts': table.iloc[1]['Pts'],
            'gd': table.iloc[1]['GD'],
            'gf': table.iloc[1]['GF']
        })
        third_placed.append({
            'team': table.iloc[2]['team'],
            'group': group_name,
            'pts': table.iloc[2]['Pts'],
            'gd': table.iloc[2]['GD'],
            'gf': table.iloc[2]['GF']
        })

    third_df = pd.DataFrame(third_placed)
    third_df = third_df.sort_values(['pts', 'gd', 'gf'], ascending=False).head(8)

    qualifiers = (
        [t['team'] for t in group_winners] +
        [t['team'] for t in group_runners_up] +
        third_df['team'].tolist()
    )

    return qualifiers, group_winners, group_runners_up, third_df

In [23]:
def simulate_knockout_rounds(teams):
    rounds = {
        'Round of 32': [],
        'Round of 16': [],
        'Quarter-finals': [],
        'Semi-finals': [],
        'Final': [],
        'Winner': None
    }

    current_round = teams.copy()
    round_names = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']

    for round_name in round_names:
        next_round = []
        np.random.shuffle(current_round)

        for i in range(0, len(current_round), 2):
            team1 = current_round[i]
            team2 = current_round[i+1]
            winner, hg, ag = simulate_knockout_match(team1, team2, neutral=True)
            next_round.append(winner)
            rounds[round_name].append(f"{team1} {hg}-{ag} {team2} → {winner}")

        current_round = next_round

    rounds['Winner'] = current_round[0]
    return rounds

# Test
knockout_results = simulate_knockout_rounds(qualifiers)

for round_name, matches in knockout_results.items():
    if round_name == 'Winner':
        print(f"\n🏆 WINNER: {matches}")
    else:
        print(f"\n{round_name}")
        for match in matches:
            print(f"  {match}")

NameError: name 'simulate_knockout_match' is not defined

In [11]:
from collections import defaultdict

N_SIMULATIONS = 10000

# Track how many times each team reaches each stage
stage_counts = defaultdict(lambda: defaultdict(int))
stages = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final', 'Winner']

print("Running 10,000 simulations...")

for sim in range(N_SIMULATIONS):
    if sim % 1000 == 0:
        print(f"  Simulation {sim}/{N_SIMULATIONS}...")

    # Simulate group stage
    all_tables = simulate_all_groups(groups, fixtures)
    qualifiers, _, _, _ = get_qualifiers(all_tables)

    # Count R32 qualifiers
    for team in qualifiers:
        stage_counts[team]['Round of 32'] += 1

    # Simulate knockouts
    knockout_results = simulate_knockout_rounds(qualifiers)

    # Count each stage
    for round_name in ['Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']:
        for match in knockout_results[round_name]:
            winner = match.split('→ ')[1].strip()
            stage_counts[winner][round_name] += 1

    # Count winner
    stage_counts[knockout_results['Winner']]['Winner'] += 1

print("Done!")

Running 10,000 simulations...
  Simulation 0/10000...


NameError: name 'simulate_knockout_match' is not defined

In [12]:
all_teams = groups['nation'].tolist()

results = []
for team in all_teams:
    row = {'Team': team}
    for stage in stages:
        count = stage_counts[team][stage]
        row[stage] = round(count / N_SIMULATIONS * 100, 1)
    results.append(row)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Winner', ascending=False).reset_index(drop=True)
results_df.index += 1
results_df

,Team,Round of 32,Round of 16,Quarter-finals,Semi-finals,Final,Winner
1,Mexico,0.0,0.0,0.0,0.0,0.0,0.0
2,South Africa,0.0,0.0,0.0,0.0,0.0,0.0
3,Iran,0.0,0.0,0.0,0.0,0.0,0.0
4,New Zealand,0.0,0.0,0.0,0.0,0.0,0.0
5,Spain,0.0,0.0,0.0,0.0,0.0,0.0
6,Cape Verde,0.0,0.0,0.0,0.0,0.0,0.0
7,Saudi Arabia,0.0,0.0,0.0,0.0,0.0,0.0
8,Uruguay,0.0,0.0,0.0,0.0,0.0,0.0
9,France,0.0,0.0,0.0,0.0,0.0,0.0
10,Senegal,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
results_df.columns = ['Team', 'R32 %', 'R16 %', 'QF %', 'SF %', 'Final %', 'Winner %']
results_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\simulation_results.csv", index=False)
print("Saved!")

Saved!
